# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruzaki11/Flyrank-ml-intern-tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/FlyRank/internship-starter/content_refresh_anonymized.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   content_id                    30000 non-null  object 
 1   client_id                     30000 non-null  object 
 2   search_volume                 27532 non-null  float64
 3   competition                   27532 non-null  float64
 4   competition_level             27390 non-null  object 
 5   cpc                           27532 non-null  float64
 6   content_type                  30000 non-null  object 
 7   main_intent                   27626 non-null  object 
 8   word_count                    22301 non-null  float64
 9   char_count                    22301 non-null  float64
 10  provider_used                 8562 non-null   object 
 11  model_used                    24267 non-null  object 
 12  impressions_90d               30000 non-null  int64  
 13  c

In [3]:
df.head(3)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False


In [4]:
df.isnull().sum()

,0
content_id,0
client_id,0
search_volume,2468
competition,2468
competition_level,2610
cpc,2468
content_type,0
main_intent,2374
word_count,7699
char_count,7699


In [5]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
    "health_score",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "trend_direction",
]

In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_pipline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([

    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))

])

preprocessor = ColumnTransformer([
    ("numeric",numeric_pipline,numeric_features),
    ("categorical",categorical_pipeline,categorical_features)
])

x = df[numeric_features + categorical_features]
x_processed = preprocessor.fit_transform(x)

I handled the missing values in the numeric feature and the categorical features , and encoded the categorical succesfully.

No additional engineered features were added in this stage.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**THE MEANINGS:**

search_volume : the number of users find the page through the search

competition : the percentage of how much this page compete the other pages of the same type of content

cpc:represents the estimated amount an advertiser pays for one click on a search

word_count: number of words in the page

char_count: number of letters in the page

impressions_90d: the number impressions over the last 90 days

clicks_90d: the number of the clicks
over the last 90 days

pageviews_90d: the number of views in the last 90 days

sessions_90d: the total number of user sessions on the content over the last 90 days

users_90d: the number of users who interact with the content over the last 90 days

engaged_sessions_90d: the number of sessions over the last 90 days where users actively engaged with the content

ai_sessions_90d: sessions over the last 90 days that are attributed to AI or automated traffic

scroll_events_90d: the number of times users performed a scroll event on the content over the last 90 days

days_with_impressions: the days that the pages got impressions on them

days_with_sessions: the days that the users perform a sessions on them

impressions_last_30d: the number of impressions on the page's content over the last 30 days

clicks_last_30d: the number of clicks that done on the page over the last 30 daya

sessions_last_30d: the total number of user sessions on the content over the last 30 days

impressions_prev_30d: the number of impressions in the previous 30 days

clicks_prev_30d: number of clicks in the previous 30 days

sessions_prev_30d: the total number of user sessions on the content in the provious 30 days

content_age_days: the age in days of the page's content since it has been uploaded

days_since_last_update: how many days passed since last update done to the page

ctr: the rate of the clicks on the page (Click-Through Rate)

avg_position: The average ranking of the content in search results.

engagement_rate: The percentage of sessions that are considered 'engaged' (e.g., meeting a minimum duration, scrolling, or specific interactions).

scroll_rate: The rate at which users scroll down the page, often indicating how much of the content is being viewed.

ai_traffic_pct: The percentage of overall traffic that is attributed to AI or automated sources.

trend_pct: The percentage change in a key metric  over a specified period, indicating its recent performance trend.

health_score: A composite score or indicator of the overall performance and quality of the content.

competition_level: A categorical assessment of the competition faced by the content in search results

content_type: The type or format of the content

main_intent: The primary goal or purpose of the user's search query that the content aims to satisfy

provider_used: The service or tool used to generate or manage the content.

model_used: The specific AI model or template used for content creation or optimization.

trend_direction: The qualitative direction of the content's performance trend

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The label-derived columns:

needs_indexing,

is_quick_win,

needs_ctr_fix,

needs_engagement_fix,

is_underperformer,

is_declining,

is_initial_refresh_candidate,

health_score,

ai_opportunity,

Future windows:

future_clicks,

future_ctr,

next_30d_sessions,


The product flags:

needs_ctr_fix

is_underperformer

ai_opportunity




**Some tests:**

In [15]:
label = df["is_initial_refresh_candidate"]
print(label.name in numeric_features + categorical_features)

False


In [17]:
suspicious_columns = [
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_underperformer",
    "is_declining",
    "health_score",
    "ai_opportunity"
]

for col in suspicious_columns:
    print(f"\n===== {col} =====")
    print(pd.crosstab(df[col], label))


===== needs_indexing =====
is_initial_refresh_candidate  False  True 
needs_indexing                            
False                         18608  11388

===== is_quick_win =====
is_initial_refresh_candidate  False  True 
is_quick_win                              
False                         17256   5855
True                           1352   5533

===== needs_ctr_fix =====
is_initial_refresh_candidate  False  True 
needs_ctr_fix                             
False                         18322  11137
True                            286    251

===== needs_engagement_fix =====
is_initial_refresh_candidate  False  True 
needs_engagement_fix                      
False                         14260   6339
True                           4348   5049

===== is_underperformer =====
is_initial_refresh_candidate  False  True 
is_underperformer                         
False                         10304  11388
True                           8304      0

===== is_declining =====
is_initial_

There is a strong relationship between these columns and the label,
but
correlation doesnot mean leakage so I will determine what to exclude next...


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

-content_id

-client_id

They identify the content/client rather than describe predictive characteristics

-is_initial_refresh_candidate

It is the prediction target.

-needs_ctr_fix

-is_underperformer

-is_declining

These have a strong correlation with the target and may cause a leakage.

-future_clicks

contain information from after the prediction point and would not be available when making the decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.